In [3]:
# Required Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Load the dataset
data = pd.read_csv('/content/SDN_DDOS_dataset.csv')

# Check column names and types
print(data.columns)

# Features and Labels split
# Rename 'label' (lowercase) if it's the target column instead of 'Label' (uppercase)
X = data.drop('label', axis=1)  # Replace 'label' if that's the correct column name for the target
y = data['label']  # Assuming 'label' is the target column for classification

# Select only numeric columns for feature set
X_numeric = X.select_dtypes(include=[np.number])

# Split dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.3, random_state=42, stratify=y
)

# Add Gaussian noise to the dataset
noise_factor = 0.1
X_train_noisy = X_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_train.shape)
X_test_noisy = X_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_test.shape)

# Clip to ensure data stays in valid range (useful if your data is bounded)
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

# Handle missing values using SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train_noisy = imputer.fit_transform(X_train_noisy)
X_test_noisy = imputer.transform(X_test_noisy)

# Scale the data using StandardScaler
scaler = StandardScaler()
X_train_noisy = scaler.fit_transform(X_train_noisy)
X_test_noisy = scaler.transform(X_test_noisy)

# Define a function to calculate and print evaluation metrics
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    print(f"\n{model_name} Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))


# 1. K-Nearest Neighbors (KNN) model
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train_noisy, y_train)
evaluate_model(knn, X_test_noisy, y_test, "K-Nearest Neighbors")

# 2. Naive Bayes model
nb = GaussianNB()
nb.fit(X_train_noisy, y_train)
evaluate_model(nb, X_test_noisy, y_test, "Naive Bayes")

# 3. Decision Tree model
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_noisy, y_train)
evaluate_model(dt, X_test_noisy, y_test, "Decision Tree")

# 4. Support Vector Machine (SVM)
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_noisy, y_train)
evaluate_model(svm, X_test_noisy, y_test, "Support Vector Machine")

# 5. Random Forest model with adjusted hyperparameters to reduce overfitting
rf = RandomForestClassifier(
    n_estimators=40,            # Fewer trees
    max_depth=4,               # Limit tree depth
    min_samples_split=8,       # Minimum samples required to split an internal node
    min_samples_leaf=5,         # Minimum samples required to be at a leaf node
    random_state=42
)
rf.fit(X_train_noisy, y_train)
evaluate_model(rf, X_test_noisy, y_test, "Random Forest")

# 6. Logistic Regression
lr = LogisticRegression(max_iter=100, random_state=42)
lr.fit(X_train_noisy, y_train)
evaluate_model(lr, X_test_noisy, y_test, "Logistic Regression")

# 7. Ensemble Model (Voting Classifier)
ensemble = VotingClassifier(estimators=[
    ('knn', knn),
    ('nb', nb),
    ('dt', dt),
    ('svm', svm),
    ('rf', rf),  # Updated Random Forest
    ('lr', lr)
], voting='hard')

ensemble.fit(X_train_noisy, y_train)
evaluate_model(ensemble, X_test_noisy, y_test, "Ensemble Voting Classifier")


# Cross-validation to check generalization performance
cv = StratifiedKFold(n_splits=5)

print("\nCross-validated KNN accuracy:", np.mean(cross_val_score(knn, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Naive Bayes accuracy:", np.mean(cross_val_score(nb, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Decision Tree accuracy:", np.mean(cross_val_score(dt, X_train_noisy, y_train, cv=cv)))
print("Cross-validated SVM accuracy:", np.mean(cross_val_score(svm, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Random Forest accuracy:", np.mean(cross_val_score(rf, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Logistic Regression accuracy:", np.mean(cross_val_score(lr, X_train_noisy, y_train, cv=cv)))
print("Cross-validated Ensemble accuracy:", np.mean(cross_val_score(ensemble, X_train_noisy, y_train, cv=cv)))


Index(['Unnamed: 0', 'time', 'switch_id', 'gfe', 'g_usip', 'rfip', 'disp_pakt',
       'disp_byte', 'mean_pkt', 'mean_byte', 'gsp', 'avg_durat',
       'avg_flow_dst', 'rate_pkt_in', 'disp_interval', 'label'],
      dtype='object')

K-Nearest Neighbors Metrics:
Accuracy: 0.9517
Precision: 0.9522
Recall: 0.9517
F1 Score: 0.9514
              precision    recall  f1-score   support

           0       0.94      0.98      0.96     15424
           1       0.96      0.90      0.93      9263

    accuracy                           0.95     24687
   macro avg       0.95      0.94      0.95     24687
weighted avg       0.95      0.95      0.95     24687


Naive Bayes Metrics:
Accuracy: 0.9123
Precision: 0.9196
Recall: 0.9123
F1 Score: 0.9133
              precision    recall  f1-score   support

           0       0.97      0.89      0.93     15424
           1       0.83      0.96      0.89      9263

    accuracy                           0.91     24687
   macro avg       0.90      0.92    